# 170 — Causal AI y descubrimiento científico

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — Escalera

a) **Peldaño 1**: compara subpoblaciones observadas; sesgo de autoselección.
b) **Peldaño 2**: es un do(mostrar tutorial) sobre toda la población.
c) **Peldaño 3**: contrafactual sobre un individuo con desenlace ya observado.
d) **Peldaño 1**: Bayes es inferencia asociativa; el posterior condiciona en el
   test observado, no interviene sobre nada.


In [ ]:
peldanos = {"a": 1, "b": 2, "c": 3, "d": 1}
print(peldanos)


## Solución 2 — Puerta trasera

**a) Intervención** (ajuste sobre Z):

```text
P(Y=1|do(X=1)) = 0.8·0.4 + 0.5·0.6 = 0.32 + 0.30 = 0.62
P(Y=1|do(X=0)) = 0.7·0.4 + 0.3·0.6 = 0.28 + 0.18 = 0.46
Efecto causal = +0.16
```

**b) Asociación**: P(X=1) = 0.7·0.4 + 0.3·0.6 = 0.46.
P(Z=1|X=1) = 0.28/0.46 ≈ 0.6087; P(Z=1|X=0) = 0.12/0.54 ≈ 0.2222.

```text
P(Y=1|X=1) = 0.8·0.6087 + 0.5·0.3913 ≈ 0.6826
P(Y=1|X=0) = 0.7·0.2222 + 0.3·0.7778 ≈ 0.3889
Diferencia asociativa ≈ +0.2937
```

**c)** La asociación (+0.29) **sobreestima** el efecto causal (+0.16): los
expertos (Z=1) usan más la función y además les va mejor de base, así que parte
de la diferencia observada es mérito de Z, no de X.


In [ ]:
p_z1 = 0.4
do_x1 = 0.8 * p_z1 + 0.5 * (1 - p_z1)
do_x0 = 0.7 * p_z1 + 0.3 * (1 - p_z1)
p_x1 = 0.7 * p_z1 + 0.3 * (1 - p_z1)
p_z1_x1 = 0.7 * p_z1 / p_x1
p_z1_x0 = 0.3 * p_z1 / (1 - p_x1)
assoc_x1 = 0.8 * p_z1_x1 + 0.5 * (1 - p_z1_x1)
assoc_x0 = 0.7 * p_z1_x0 + 0.3 * (1 - p_z1_x0)
print(f"do: {do_x1:.3f} vs {do_x0:.3f}  (efecto {do_x1-do_x0:+.3f})")
print(f"asociación: {assoc_x1:.4f} vs {assoc_x0:.4f}  (dif {assoc_x1-assoc_x0:+.4f})")
assert (assoc_x1 - assoc_x0) > (do_x1 - do_x0)


## Solución 3 — Bayes es peldaño 1

a) Bayes: 0.3214 = 0.05·0.9 / (0.05·0.9 + 0.95·fpr) →
0.045 + 0.95·fpr·0.3214 = ... despejando: denominador = 0.045/0.3214 = 0.14, así
que 0.95·fpr = 0.095 y **fpr = 0.10**. El laboratorio usa una tasa de falsos
positivos del 10 %.

b) El posterior describe la subpoblación observada "test positivo": es
P(enfermedad | test), asociación. No dice qué pasaría al intervenir (p. ej.,
si tratar a todos los positivos cambia desenlaces, o si el test *causa* algo).
Para preguntas do() haría falta un grafo causal del proceso test-enfermedad-
tratamiento y datos o supuestos adicionales.


In [ ]:
result = run_lab("probability", seed=170)
r = result["result"]
fpr = 0.10
posterior = r["prior"] * r["likelihood"] / (
    r["prior"] * r["likelihood"] + (1 - r["prior"]) * fpr)
print(f"posterior reconstruido: {posterior:.4f}  (lab: {r['posterior']})")
assert abs(posterior - r["posterior"]) < 1e-3


## Solución 4 — Colisionador

Entre los admitidos, saber que alguien tiene poco talento *informa* que
probablemente compensó con mucho esfuerzo (y viceversa): condicionar en el
colisionador Admisión=1 crea dependencia entre sus causas, que eran
independientes. Es el sesgo de selección clásico ("en esta universidad los
deportistas sacan peores notas").

La regla violada: el conjunto de ajuste de puerta trasera **no debe contener
colisionadores del camino** (ni descendientes de X): condicionar en un
colisionador *abre* el camino Talento → Admisión ← Esfuerzo en lugar de
bloquearlo. Ajustar por más variables no es gratis; el grafo dice cuáles sí.
